In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from ucimlrepo import fetch_ucirepo

In [ ]:
# =========================================================
# 07-3 재현 및 검증
# =========================================================

tf.keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)


# model_fn 사용 여부
def model_fn():
    model = Sequential([
        Input(shape=(28, 28)),
        Flatten(),
        Dense(100, activation='relu'),
        Dropout(0.3),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


model = model_fn()

# restore_best_weights=True 설정
early_stopping_cb = EarlyStopping(
    patience=3,
    restore_best_weights=True
)

# callbacks 리스트 전달 확인
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping_cb]
)

val_loss, val_accuracy = model.evaluate(X_val, y_val)

print("stopped_epoch:", early_stopping_cb.stopped_epoch)
print("val_accuracy:", val_accuracy)

# Dropout(0.3)+Adam+EarlyStopping, stopped_epoch 출력, val_accuracy ≥ 0.8853
print("val_accuracy >= 0.8853:", val_accuracy >= 0.8853)

Epoch 1/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7888 - loss: 0.5964 - val_accuracy: 0.8518 - val_loss: 0.4131
Epoch 2/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8405 - loss: 0.4415 - val_accuracy: 0.8665 - val_loss: 0.3706
Epoch 3/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8525 - loss: 0.4070 - val_accuracy: 0.8718 - val_loss: 0.3539
Epoch 4/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8584 - loss: 0.3859 - val_accuracy: 0.8762 - val_loss: 0.3410
Epoch 5/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8641 - loss: 0.3700 - val_accuracy: 0.8798 - val_loss: 0.3343
Epoch 6/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8680 - loss: 0.3555 - val_accuracy: 0.8801 - val_loss: 0.3300
Epoch 7/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8718 - loss: 0.3459 - val_accuracy: 0.8810 - val_loss: 0.3279
Epoch 8/30
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8744 - loss: 0.3394 - 

In [5]:
# =========================================================
# Dropout Rate 실험
# =========================================================

dropout_rates = [0.0, 0.2, 0.3, 0.5]
dropout_results = []


def model_fn_dropout(dropout_rate):
    model = Sequential([
        Input(shape=(28, 28)),
        Flatten(),
        Dense(100, activation='relu'),
        Dropout(dropout_rate),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


for rate in dropout_rates:
    tf.keras.utils.set_random_seed(42)

    model = model_fn_dropout(rate)

    early_stopping_cb = EarlyStopping(
        patience=3,
        restore_best_weights=True
    )

    # 실험 조건 동일
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=32,
        callbacks=[early_stopping_cb],
        verbose=0
    )

    val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)

    dropout_results.append({
        "dropout_rate": rate,
        "stopped_epoch": early_stopping_cb.stopped_epoch,
        "val_accuracy": val_accuracy
    })


# 결과표 작성
df_dropout = pd.DataFrame(dropout_results)
display(df_dropout)

# 4가지 rate 비교표 + "rate=0.5 관찰" 주석
print("rate=0.5는 학습 중 은닉층 뉴런의 절반을 무작위로 꺼서 과적합을 줄일 수 있지만, 너무 많은 정보를 버리기 때문에 정확도가 낮아질 수도 있다.")

,dropout_rate,stopped_epoch,val_accuracy
0,0.0,10,0.883167
1,0.2,10,0.885667
2,0.3,16,0.890833
3,0.5,12,0.881917


rate=0.5는 학습 중 은닉층 뉴런의 절반을 무작위로 꺼서 과적합을 줄일 수 있지만, 너무 많은 정보를 버리기 때문에 정확도가 낮아질 수도 있다.


In [6]:
# =========================================================
# Dry Bean 데이터 MLP vs R
# =========================================================

dry_bean = fetch_ucirepo(id=602)

X = dry_bean.data.features
y = dry_bean.data.targets

print(X.shape)
print(y.shape)
display(X.head())
display(y.head())


# LabelEncoder 사용
le = LabelEncoder()
y_encoded = le.fit_transform(y.values.ravel())

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# StandardScaler 적용
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


mlp_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

mlp_model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping_cb = EarlyStopping(
    patience=5,
    restore_best_weights=True
)

mlp_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping_cb],
    verbose=0
)

mlp_loss, mlp_acc = mlp_model.evaluate(X_val_scaled, y_val, verbose=0)


rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_val_scaled)
rf_acc = accuracy_score(y_val, rf_pred)


df_compare = pd.DataFrame({
    "모델": ["MLP", "RandomForest"],
    "val_accuracy": [mlp_acc, rf_acc]
})

display(df_compare)

(13611, 16)
(13611, 1)


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRatio,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,Roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166


,Class
0,SEKER
1,SEKER
2,SEKER
3,SEKER
4,SEKER


,모델,val_accuracy
0,MLP,0.923246
1,RandomForest,0.920676
